# Checkpoint и resume QAOA

Этот ноутбук показывает:
- разницу между новым `solve()` и продолжением `resume()`;
- согласованную паузу после двух ADAM-итераций;
- inspect без запуска solver;
- продолжение в новом объекте и сравнение с непрерывным запуском;
- консервативный charge после незавершённой попытки;
- capability matrix для остальных реализаций.

Основной пример — CPU и малый statevector. GPU checkpoint в `0.1.0a10`
намеренно не заявлен: CuPy RNG требует отдельного переносимого протокола.


In [1]:
from copy import deepcopy
from pathlib import Path
import os
import tempfile

ROOT = Path.cwd()
if not (ROOT / "experiments").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from QA_max_cut_lib import GraphFactory, QAOACPU, RunControl, inspect_checkpoint, restore, tools
from QA_max_cut_lib.core.checkpoint import ResourceJournal
from QA_max_cut_lib.core.checkpoint_capabilities import capabilities

graph = GraphFactory.from_recipe(tools.load_json("configs/graphs/complete_signed_7.json"))
base_config = tools.load_json("configs/solvers/qaoa_cpu_checkpoint.json")
root = Path(tempfile.mkdtemp(prefix="qa_a10_notebook_"))
checkpoint_config = deepcopy(base_config)
checkpoint_config["checkpoint"]["path"] = str(root)
checkpoint_config["output"] = {"report": 0, "logs": 0, "color": "never"}
print("store:", root)


store: C:\Users\Alexey\AppData\Local\Temp\qa_a10_notebook_w_2rnqdc


## A. Непрерывный запуск и пауза

Непрерывный и паузный запуски используют одну и ту же конфигурацию и seed.
Ожидаемые training shots: 320 после двух итераций, held readout: 64.


In [2]:
continuous_config = deepcopy(checkpoint_config)
continuous_config.pop("checkpoint", None)
continuous = QAOACPU(continuous_config)
continuous_solution = continuous.solve(graph)
print("continuous:", continuous.status, continuous.resources["shots_used"], continuous.cut_value)

paused = QAOACPU(checkpoint_config)
paused_solution = paused.solve(graph, control=RunControl(pause_after={"optimizer_iterations": 2}))
print("paused:", paused.status, paused.termination_reason)
print("resources:", paused.resources)
print("checkpoint:", paused.last_checkpoint.path)
info = inspect_checkpoint(paused.last_checkpoint.path)
print("phase:", info["phase"], "next:", info["next_operation"])
assert paused.status == "paused"
assert paused.resources["known_spent"] == 320
assert paused.resources["readout_reserved"] == 64


continuous: completed 704 2.0
paused: paused checkpoint_pause
resources: {'shots_used': 320, 'known_spent': 320, 'uncertain_upper_charge': 0, 'total_charged': 320, 'training_shots': 320, 'logical_training_shots': 320, 'readout_shots': 0, 'readout_reserved': 64, 'shots_available': 320, 'objective_evaluations': 10, 'optimizer_steps': 2, 'num_stages': 3, 'num_treads': 1, 'checkpoint_writes': 2}
checkpoint: C:\Users\Alexey\AppData\Local\Temp\qa_a10_notebook_w_2rnqdc\1cab7ad4-275c-4fb0-92b7-30c2afe506b9\gen_000002
phase: training next: {'optimizer_iteration': 3}


## B. Продолжение в новом объекте

`restore()` не запускает solver. Только `resume()` продолжает canonical run.


In [3]:
continued = restore(paused.last_checkpoint.path)
final_solution = continued.resume()
print("resumed:", continued.status, continued.resources["shots_used"], continued.cut_value)
print("same run id:", continued.run_id == paused.run_id)
print("same solution:", final_solution == continuous_solution)
print("max angle error:", float(np.max(np.abs(continued.optimal_angles - continuous.optimal_angles))))
assert final_solution == continuous_solution
assert np.allclose(continued.optimal_angles, continuous.optimal_angles)
assert continued.resources["objective_evaluations"] == continuous.resources["objective_evaluations"]
assert continued.resources["shots_used"] == continuous.resources["shots_used"] == 704


resumed: completed 704 2.0
same run id: True
same solution: True
max angle error: 0.0


## C. Незавершённая попытка и консервативный charge

BEGIN без COMMIT/RESOLVE означает, что измерение могло состояться. Журнал
списывает upper bound один раз и закрывает операцию. Это не выдаётся за
доказанное число физических shots.


In [4]:
root_b = Path(tempfile.mkdtemp(prefix="qa_a10_b_"))
config_b = deepcopy(checkpoint_config)
config_b["checkpoint"]["path"] = str(root_b)
solver_b = QAOACPU(config_b)
solver_b.solve(graph, control=RunControl(pause_after={"optimizer_iterations": 2}))
run_dir = Path(solver_b.last_checkpoint.path).parent
journal = ResourceJournal(run_dir)
journal.begin(f"{solver_b.run_id}:iteration:3", attempt_id="crash-demo", upper_bound=160, kind="optimizer_iteration")
recovered = restore(solver_b.last_checkpoint.path)
recovered.resume()
print("recovered:", recovered.status, recovered.resources)
assert recovered.resources["known_spent"] == 544
assert recovered.resources["uncertain_upper_charge"] == 160
assert recovered.resources["total_charged"] == 704


recovered: stopped {'shots_used': 704, 'known_spent': 544, 'uncertain_upper_charge': 160, 'total_charged': 704, 'training_shots': 480, 'logical_training_shots': 480, 'readout_shots': 64, 'readout_reserved': 0, 'shots_available': 0, 'objective_evaluations': 15, 'optimizer_steps': 3, 'num_stages': 4, 'num_treads': 1, 'checkpoint_writes': 5}


## D. Capability matrix

Неподдержанный компонент отклоняется явно; он не превращается в фиктивный
частичный checkpoint.


In [5]:
table = capabilities()
for name in ("qaoa_cpu", "qaoa_gpu", "rqaoa_cpu", "ls_rqaoa_cpu", "qaoa_in_qaoa_cpu", "exact_cpu", "gw_cpu", "simulated_annealing_cpu"):
    item = table[name]
    print(f"{name:24s} supported={item['supports_checkpoint']!s:5s} boundaries={item['boundaries']}")


qaoa_cpu                 supported=True  boundaries=['completed ADAM iteration', 'training_done', 'readout_done']
qaoa_gpu                 supported=False boundaries=[]
rqaoa_cpu                supported=False boundaries=[]
ls_rqaoa_cpu             supported=False boundaries=[]
qaoa_in_qaoa_cpu         supported=False boundaries=[]
exact_cpu                supported=False boundaries=[]
gw_cpu                   supported=False boundaries=[]
simulated_annealing_cpu  supported=False boundaries=[]


## E. Размер поколений и бюджеты

Checkpoint содержит manifest, state.json и четыре числовых `.npy` sidecar для
`theta/moment/variance/best_theta`. Большой statevector по умолчанию не
сохраняется.


In [6]:
run_dir = Path(continued.last_checkpoint.path).parent
gen_sizes = []
for gen in sorted(run_dir.glob("gen_*")):
    size = sum(p.stat().st_size for p in gen.iterdir() if p.is_file())
    gen_sizes.append((gen.name, size))
print("generations:", gen_sizes)
print("total bytes:", sum(v for _, v in gen_sizes))


generations: [('gen_000001', 8612), ('gen_000002', 10282), ('gen_000003', 12044), ('gen_000004', 13683), ('gen_000005', 13686), ('gen_000006', 13268)]
total bytes: 71575


## F. График lineage и расхода

График строится по сохранённым manifests; solver не вызывается.


In [7]:
import json
manifest_rows = []
for gen in sorted(run_dir.glob("gen_*")):
    data = json.loads((gen / "manifest.json").read_text(encoding="utf-8"))
    manifest_rows.append((data["generation"], data.get("phase"), data.get("journal_cutoff", {}).get("known_spent", 0)))
fig, ax = plt.subplots(figsize=(9, 4))
xs = [row[0] for row in manifest_rows]
ys = [row[2] for row in manifest_rows]
ax.step(xs, ys, where="post", marker="o", color="#2563eb")
ax.set_xlabel("generation")
ax.set_ylabel("known training shots")
ax.set_title("Checkpoint lineage and accepted resource prefix")
ax.grid(alpha=0.25)
fig.tight_layout()
figure_path = Path("docs/notebook_previews/11_checkpoints_and_resume.png")
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=140, facecolor="white")
plt.close(fig)
print("saved:", figure_path)


saved: docs\notebook_previews\11_checkpoints_and_resume.png


## Итог

- `solve()` всегда новый запуск.
- pause → inspect → restore → resume продолжает тот же run.
- После двух итераций: 320 training shots, 64 held, 320 available.
- Completed readout не повторяет M измерений.
- При аварии uncertain charge не равен нулю и не удваивается при reconciliation.
- GPU/RQAOA/LS/hierarchy checkpoint в a10 явно не заявлены и требуют
  соответствующих codecs.
